## A Deep research Rough Work

In [1]:
%%html
<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;\" > Commercial implications</h2>
            <span style="color:#00bfff;\">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>


,"Commercial implications A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!"


In [2]:
# Importing packages 

# uv add ddgs for web search

from agents import (
    Agent,
    WebSearchTool,
    trace,
    Runner,
    gen_trace_id,
    function_tool,
    set_tracing_export_api_key
)
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import smtplib
import traceback
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os
import litellm
from typing import Dict
from IPython.display import display, Markdown
from ddgs import DDGS

# Suppress LiteLLM verbose messages
litellm.set_verbose = False
litellm.suppress_debug_info = True

python-dotenv could not parse statement starting at line 13
python-dotenv could not parse statement starting at line 13


In [3]:
load_dotenv(override=True)



python-dotenv could not parse statement starting at line 13


True

## OpenAI Hosted Tools

OpenAI Agents SDK includes the following hosted tools:

The `WebSearchTool` lets an agent search the web.  
The `FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
The `ComputerTool` allows automating computer use tasks like taking screenshots and clicking.

### Important note - API charge of WebSearchTool

This is costing me 2.5 cents per call for OpenAI WebSearchTool. That can add up to $2-$3 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 2.5 cents per call.

Costs are here: https://platform.openai.com/docs/pricing#web-search

In [4]:
import os
from openai import OpenAI

# Correct OpenRouter Setup
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = "https://openrouter.ai/api/v1"
model1 = "google/gemini-2.0-flash-001"
model2 = "openai/gpt-4o-mini"

client = OpenAI(api_key=api_key, base_url=base_url)


In [5]:
import os
from agents import (
    Agent,
    Runner,
    enable_verbose_stdout_logging,
    function_tool,
    set_default_openai_api,
    set_default_openai_client,
    set_tracing_disabled,
    set_tracing_export_api_key,
    trace,
)
from openai import AsyncOpenAI

# Configure default SDK client to use OpenRouter
api_key = os.getenv("OPENROUTER_API_KEY")
base_url = "https://openrouter.ai/api/v1"
client = AsyncOpenAI(api_key=api_key, base_url=base_url)

set_default_openai_api("chat_completions")
set_default_openai_client(client)

# Using standard OpenAI Key for tracing
set_tracing_export_api_key(os.getenv("OPENAI_API_KEY"))
enable_verbose_stdout_logging()


In [6]:
@function_tool
def search_the_web(query: str) -> str:
    """Searches the web for the given query and returns a summary of results."""
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=2):
            results.append(f"Title: {r['title']}\nURL: {r['href']}\nSnippet: {r['body']}\n")
    return "\n---\n".join(results)

INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."


set_tracing_export_api_key(os.getenv('OPENAI_API_KEY'))


search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[search_the_web],
    model="litellm/openrouter/google/gemini-2.0-flash-001",
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)
    

display(Markdown(result.final_output))

Creating trace Search with id trace_637d987a77f14307b6f90f2a4e4b498b
Setting current trace: trace_637d987a77f14307b6f90f2a4e4b498b
Creating span <agents.tracing.span_data.AgentSpanData object at 0x000001FB615A2C10> with id None
Running agent Search agent (turn 1)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB615B43B0> with id None
Calling LLM
Received model response
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x000001FB61DED590> with id None
Invoking tool search_the_web
Exported 2 items
Tool search_the_web completed.
Running agent Search agent (turn 2)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB623823F0> with id None
Calling LLM
Received model response
Resetting current trace


AI agent frameworks in 2025, top frameworks include LangGraph, DSPy, CrewAI, and Agno. Agentbase, AutoGen, and LangChain are also mentioned as top frameworks for building production-ready AI agents. The frameworks are compared for developers. A complete developer guide to the best AI agent frameworks in 2025 is available.


Exported 7 items
Exported 2 items
Exported 1 items
Exported 3 items
Exported 2 items
Exported 2 items
Exported 2 items
Exported 2 items


In [8]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 1

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="litellm/openrouter/google/gemini-3.1-flash-lite-preview",
    output_type=WebSearchPlan,
)

In [9]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

Creating trace Search with id trace_89d1000893104e1baf447697033907bb
Setting current trace: trace_89d1000893104e1baf447697033907bb
Creating span <agents.tracing.span_data.AgentSpanData object at 0x000001FB6158F0C0> with id None
Running agent PlannerAgent (turn 1)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB623827B0> with id None
Calling LLM
Received model response
searches=[WebSearchItem(reason='Identify the most prominent and high-growth AI agent development frameworks trending in 2025 to provide a current and comprehensive overview.', query='best AI agent frameworks 2025 overview')]
Resetting current trace


In [10]:
@function_tool
def send_email(subject: str, html_body: str, recipient: str = None) -> str:
    """Send an email with the given subject and body. Returns 'success' or error message."""
    try:    
        # Use recipient if provided, else default to environment variable
        email_user = os.environ.get('EMAIL_USER')
        email_password = os.environ.get('EMAIL_PASSWORD')
        target_recipient = recipient or email_user

        if not email_user or not email_password:
            return "Error: EMAIL_USER or EMAIL_PASSWORD not set in environment."

        msg = MIMEMultipart()
        msg['From'] = f"Omnicom Solutions <{email_user}>"
        msg['To'] = target_recipient
        msg['Subject'] = subject
        msg.attach(MIMEText(html_body, 'text/html')) 
        
        # SSL (port 465) is much more reliable for Gmail
        with smtplib.SMTP_SSL('smtp.gmail.com', 465, timeout=12) as server:
            server.login(email_user, email_password)
            server.send_message(msg)
        
        return "success"
    except Exception as e:
        return f"Error sending email: {str(e)}"



In [11]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=model2, # gpt-4o-mini via OpenRouter
)


In [12]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=model2, # gpt-4o-mini via OpenRouter
    output_type=ReportData,
)


### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [13]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [14]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### SHOW TIME

In [15]:
from IPython.display import display, HTML, Markdown

query = "Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    
    # ── Minimal HTML Output ──
    display(HTML(f"<h2>&#128269; Research Results: {query}</h2>"))
    
    summary_box = f"""
    <div style='background:#f8f9fa; border:1px solid #dee2e6; border-radius:5px; padding:15px; margin:15px 0;'>
        <h3 style='margin-top:0;'>&#128203; Executive Summary</h3>
        <p style='line-height:1.6;'>{report.short_summary}</p>
    </div>
    """
    display(HTML(summary_box))
    
    display(Markdown(report.markdown_report))
    
    if report.follow_up_questions:
        qs = "".join([f"<li>{q}</li>" for q in report.follow_up_questions])
        qs_box = f"""
        <div style='background:#e9ecef; border-radius:5px; padding:15px; margin:15px 0;'>
            <h3 style='margin-top:0;'>&#128270; Follow-Up Questions</h3>
            <ul>{qs}</ul>
        </div>
        """
        display(HTML(qs_box))
    
    # ── Send Email ──
    await send_email(report)  
    print("Hooray! Report displayed and email sent.")


Creating trace Research trace with id trace_d9be84cc738445888506d5cb3bc47d9e
Setting current trace: trace_d9be84cc738445888506d5cb3bc47d9e
Starting research...
Planning searches...
Creating span <agents.tracing.span_data.AgentSpanData object at 0x000001FB61CAACB0> with id None
Running agent PlannerAgent (turn 1)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB61C9EB70> with id None
Calling LLM
Received model response
Will perform 1 searches
Searching...
Creating span <agents.tracing.span_data.AgentSpanData object at 0x000001FB61CAA670> with id None
Running agent Search agent (turn 1)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB61C9F770> with id None
Calling LLM
Received model response
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x000001FB615A2C10> with id None
Invoking tool search_the_web
Tool search_the_web completed.
Running agent Search agent (turn 2)
Creating span <agents.tracing.span_data.Gene

# Report on Latest AI Agent Frameworks in 2025

## Table of Contents
1. **Introduction**  
   1.1. Purpose of the Report  
   1.2. Definition of AI Agent Frameworks  

2. **Overview of AI Agent Frameworks**  
   2.1. Importance in Modern Applications  
   2.2. Types of AI Agent Frameworks  

3. **Leading AI Agent Frameworks**  
   3.1. LangChain  
   3.2. CrewAI  
   3.3. LangGraph  
   3.4. AutoGen  
   3.5. AgentGPT  
   3.6. AutoGPT  

4. **Comparison of AI Agent Frameworks**  
   4.1. Suitability for Startups and SMBs  
   4.2. Suitability for Enterprise Applications  
   4.3. Rapid Prototyping and Accessibility  
   4.4. Adoption Trends  

5. **Future Prospects and Trends**  
   5.1. Evolving Features and Innovations  
   5.2. Market Growth and Opportunities  
   5.3. Challenges in the Landscape  

6. **Conclusion**  
   6.1. Summary of Findings  
   6.2. Recommendations for Stakeholders  

7. **Follow-Up Questions**  

## 1. Introduction  
### 1.1. Purpose of the Report  
This report aims to explore the latest frameworks for AI agents as of 2025, providing insights into their functionalities, target user groups, and industry implications. The rapid evolution of artificial intelligence necessitates ongoing assessment to keep track of which frameworks are gaining traction and why.

### 1.2. Definition of AI Agent Frameworks  
AI agent frameworks serve as the infrastructure and tools that allow developers to create, deploy, and manage intelligent agents capable of interacting with users, processing information, and learning from data. They are integral to building applications that leverage AI capabilities, often enhancing automation and decision-making processes.

## 2. Overview of AI Agent Frameworks  
### 2.1. Importance in Modern Applications  
AI agents are increasingly used across various sectors, including customer service, healthcare, finance, and beyond. They drive innovations in user experience, efficiency, and scalability, allowing businesses to optimize their services and reach.

### 2.2. Types of AI Agent Frameworks  
AI agent frameworks can be categorized based on complexity and target audience:  
- **Basic Frameworks**: For rapid application development suitable for startups and individual developers.  
- **Advanced Frameworks**: Tailored for complex and enterprise-level applications requiring robust solutions.  
- **Specialized Frameworks**: Focused on specific industries or applications, offering tailored solutions.

## 3. Leading AI Agent Frameworks  
### 3.1. LangChain  
LangChain is recognized for its versatile capabilities, making it a favorite among developers seeking to implement AI agents effectively. Its adoption continues to remain high in 2025, largely attributed to ongoing support, extensive documentation, and a strong community.  

### 3.2. CrewAI  
CrewAI positions itself as a user-friendly framework aimed at startups and SMBs. Its functionality facilitates real-time collaboration and integration with existing services, thereby enhancing user engagement and productivity.  

### 3.3. LangGraph  
LangGraph distinguishes itself by providing graph-based data modeling tools alongside AI capabilities. This makes it an excellent choice for companies that require sophisticated data analysis in their AI applications, particularly in enterprise settings.  

### 3.4. AutoGen  
AutoGen stands out due to its ability to automatically generate code for various AI applications, which accelerates deployment and reduces time-to-market. The framework is experiencing significant growth among developers, reflecting its ease of use and innovative features.  

### 3.5. AgentGPT  
AgentGPT is designed with accessibility in mind. It offers a straightforward interface and quick setup, making it ideal for those embarking on rapid prototyping. The focus on user-friendly development makes it a go-to tool for hackathons and educational purposes.  

### 3.6. AutoGPT  
AutoGPT also emphasizes rapid prototyping, catering to smaller teams and solo developers. Its lightweight nature allows users to experiment with AI functionalities without extensive investment in time and resources.  

## 4. Comparison of AI Agent Frameworks  
### 4.1. Suitability for Startups and SMBs  
Frameworks like CrewAI and AgentGPT provide the necessary support for startups and SMBs. These frameworks prioritize user assistance, lower technical barriers, and affordable solutions, fostering innovation in small organizations.

### 4.2. Suitability for Enterprise Applications  
LangChain, LangGraph, and AutoGen are geared towards enterprises requiring extensive customization and superior performance. These frameworks often come with advanced features that meet the rigorous demands of large organizations.

### 4.3. Rapid Prototyping and Accessibility  
The growth of AgentGPT and AutoGPT highlights the increasing need for accessible tools that allow for quick experimentation. These tools are particularly valuable as they facilitate innovation cycles, enabling faster validation of ideas before implementation.

### 4.4. Adoption Trends  
The adoption rate for frameworks like LangChain and LangGraph remains high, reflective of their stability and performance reliability. In contrast, the rapid increase in AutoGen's popularity points towards a market trend favoring automation and ease of use.

## 5. Future Prospects and Trends  
### 5.1. Evolving Features and Innovations  
In the coming years, AI agent frameworks are likely to evolve significantly with enhancements in machine learning capabilities, natural language processing, and real-time data processing, making them even more intelligent and autonomous.

### 5.2. Market Growth and Opportunities  
The market for AI agent frameworks is expected to expand as more organizations adopt AI strategies. This growth will foster competition among existing frameworks and new entrants, driving innovation and enhancements in service offerings.

### 5.3. Challenges in the Landscape  
Despite the promising trends, challenges such as integration complexities, security concerns, and the need for ongoing skill development remain prevalent. Frameworks that address these challenges effectively will likely position themselves favorably in the competitive landscape.

## 6. Conclusion  
### 6.1. Summary of Findings  
In 2025, the landscape of AI agent frameworks is characterized by both established players like LangChain and new offerings such as AutoGen. Startups and SMBs are well-served by accessible frameworks, while enterprises have access to more specialized and powerful solutions.

### 6.2. Recommendations for Stakeholders  
Stakeholders in the field of AI development should aim to stay current with the evolving frameworks, consider the specific needs of their applications, and prioritize frameworks that enhance productivity and innovation.

## 7. Follow-Up Questions  
- What are the emerging trends in AI agent frameworks beyond 2025?  
- How do different industries utilize these frameworks?  
- What key innovations are anticipated in AI and related technologies?  
- What are user experiences and feedback for these frameworks?  
- How are organizations managing the integration of AI agents within existing systems?

Writing email...
Creating span <agents.tracing.span_data.AgentSpanData object at 0x000001FB624768A0> with id None
Running agent Email agent (turn 1)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB62382390> with id None
Calling LLM
Received model response
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x000001FB5C5F7430> with id None
Invoking tool send_email
Tool send_email completed.
Running agent Email agent (turn 2)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB61C9EB70> with id None
Calling LLM
Received model response
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x000001FB5AB66030> with id None
Invoking tool send_email
Tool send_email completed.
Running agent Email agent (turn 3)
Creating span <agents.tracing.span_data.GenerationSpanData object at 0x000001FB62382390> with id None
Calling LLM
Received model response
Email sent
Hooray! Report displayed and email sent.
Resetting c